# Week 3 Laboratory
# Task 2: Image Classification using K-Nearest Neighbours (KNN)

---

## Learning Objectives

After completing this laboratory, you should be able to:

✓ Load image datasets and extract 3D color histogram features using OpenCV.

✓ Split a dataset into **Training (60%)**, **Validation (20%)**, and **Test (20%)** sets.

✓ Perform hyperparameter tuning on a validation set to select the optimal $k$ with early stopping.

✓ Evaluate a KNN classifier using Accuracy, Precision, Recall, F1-score, and Confusion Matrix.

✓ Benchmark model inference speed and perform qualitative error analysis on image predictions.

---
## Setup Environment

Install necessary packages to run this notebook.

In [ ]:
!pip install opencv-python numpy matplotlib tqdm scikit-learn pandas

---
## 1. Imports & Pre-defined Helper Functions

We first import the required packages and provide four utility functions:
* `preprocess_image`: Reads and resizes an image to $150 \times 150$.
* `extract_color_histogram`: Computes normalized 3D color histograms ($6 \times 6 \times 6 = 216$ bins).
* `load_dataset`: Loads images from `flowers/` across all 5 categories (`daisy`, `dandelion`, `rose`, `sunflower`, `tulip`).
* `show_images`: Displays correctly or incorrectly classified images with titles.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from timeit import default_timer as timer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

import warnings
warnings.filterwarnings('ignore')

LABELS = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

def preprocess_image(path_to_image, img_size=150):
    """Read and resize an input image."""
    img = cv2.imread(path_to_image, cv2.IMREAD_COLOR)
    img = cv2.resize(img, (img_size, img_size))
    return np.array(img)

def extract_color_histogram(dataset, hist_size=6):
    """Extract color histogram features from a dataset of images."""
    col_hist = []
    for img in dataset:
        hist = cv2.calcHist([img], [0, 1, 2], None, (hist_size, hist_size, hist_size), [0, 256, 0, 256, 0, 256])
        col_hist.append(cv2.normalize(hist, None, 0, 1, cv2.NORM_MINMAX).flatten())
    return np.array(col_hist)

def load_dataset(base_path='flowers'):
    """Load dataset images and labels."""
    X, Y = [], []
    for i in range(len(LABELS)):
        current_size = len(X)
        dir_path = os.path.join(base_path, LABELS[i])
        if os.path.exists(dir_path):
            for img in tqdm(os.listdir(dir_path), desc=f"Loading {LABELS[i]}"):
                if not img.startswith('.'):
                    X.append(preprocess_image(os.path.join(dir_path, img)))
                    Y.append(LABELS[i])
            print(f'Loaded {len(X) - current_size} {LABELS[i]} images')
    return X, Y

def show_images(ground_truth, predictions, img_test, correct=True, num_images=5):
    """Show correctly or incorrectly classified images."""
    count = 0
    plt.figure(figsize=(15, 3))
    for i in range(len(ground_truth)):
        if (ground_truth[i] == predictions[i]) == correct:
            count += 1
            plt.subplot(1, num_images, count)
            plt.imshow(cv2.cvtColor(img_test[i], cv2.COLOR_BGR2RGB))
            plt.title(f"GT: {ground_truth[i]}\nPred: {predictions[i]}", fontsize=9)
            plt.axis('off')
            if count == num_images:
                break
    plt.tight_layout()
    plt.show()

print("Environment set up successfully!")

---
## STEP 1. Load Dataset

Load the flower image dataset into memory using `load_dataset()`.

In [ ]:
# STEP 1. Load dataset
# YOUR CODE HERE
X, Y = load_dataset('flowers')
print(f"Total samples loaded: {len(X)}")

---
## STEP 2. Split Dataset (Train, Validation, Test)

We split the dataset into three subsets:
* **Train Set (60%)**: Fits the KNN model.
* **Validation Set (20%)**: Tunes the hyperparameter $k$.
* **Test Set (20%)**: Held-out set for final model evaluation.

### Task:
Use `train_test_split` twice:
1. Reserve 20% of data for validation (`img_val`, `y_val`).
2. Reserve 20% of original data (25% of remaining) for testing (`img_test`, `y_test`).

In [ ]:
# STEP 2. Split dataset into train, validation, and test sets
# YOUR CODE HERE
# img_train, img_val, y_train, y_val = ...
# img_train, img_test, y_train, y_test = ...

print(f"Training samples  : {len(img_train)} ({len(img_train)/len(X)*100:.1f}%)")
print(f"Validation samples: {len(img_val)} ({len(img_val)/len(X)*100:.1f}%)")
print(f"Test samples      : {len(img_test)} ({len(img_test)/len(X)*100:.1f}%)")

---
## STEP 3. Feature Extraction

Extract 3D color histogram feature vectors ($216$ dimensions per image) from the training, validation, and test image sets.

### Task:
Apply `extract_color_histogram` to `img_train`, `img_val`, and `img_test` to produce `x_train`, `x_val`, and `x_test`.

In [ ]:
# STEP 3. Extract colour histogram features from the datasets
# YOUR CODE HERE
# x_train = ...
# x_val = ...
# x_test = ...

print(f"x_train shape: {x_train.shape}")
print(f"x_val shape  : {x_val.shape}")
print(f"x_test shape : {x_test.shape}")

---
## STEP 4. Determine Optimal Value of $k$

Search for the optimal $k$ by training `KNeighborsClassifier(n_neighbors=k)` on `x_train` and evaluating accuracy on `x_val`.

### Early Stopping
To optimize search speed, stop early if validation accuracy fails to improve for `EARLY_STOP = 100` consecutive steps.

### Task:
Complete the hyperparameter tuning loop below.

In [ ]:
# STEP 4. Determine the optimal values of k
validation_performance = []
best_index = 0
optimal_k = 1
EARLY_STOP = 100
K_VALUES = list(range(1, len(x_train) + 1))

# YOUR CODE HERE
# Loop through K_VALUES:
# 1. Fit KNeighborsClassifier(n_neighbors=k) on x_train, y_train
# 2. Predict on x_val and calculate val_accuracy
# 3. Update best_index and optimal_k if val_accuracy improves
# 4. Break early if len(validation_performance) - best_index > EARLY_STOP

print(f"The optimal value of k is: {optimal_k} (Val Acc: {validation_performance[best_index]*100:.2f}%)")

Let's plot Validation Accuracy vs. $k$ to visualize the search.

In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(K_VALUES[:len(validation_performance)], [acc * 100 for acc in validation_performance], color='navy', label='Validation Accuracy')
plt.axvline(x=optimal_k, color='red', linestyle='--', label=f'Optimal k = {optimal_k}')
plt.title('Validation Accuracy vs. k', fontsize=12)
plt.xlabel('k (Number of Neighbors)', fontsize=10)
plt.ylabel('Validation Accuracy (%)', fontsize=10)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

---
## STEP 5. Train KNN Classifier with Optimal $k$

Train the final `KNeighborsClassifier` using `n_neighbors=optimal_k` on `x_train` and `y_train`.

In [ ]:
# STEP 5. Train a k-NN classifier with the optimal k
# YOUR CODE HERE
# knn_model = ...
# knn_model.fit(...)

print("Model trained successfully!")

---
## STEP 6. Benchmark Inference Speed on Test Set

Measure the average inference time per test image over 10 runs using `timer()`.

In [ ]:
# STEP 6. Evaluate the k-NN on the test dataset
inference_times = []
# YOUR CODE HERE
# Loop 10 times: record start_time, perform y_pred = knn_model.predict(x_test), record end_time
# Append (end_time - start_time) / len(x_test) to inference_times

average_inference_time = np.mean(inference_times)
print(f"Average inference time: {average_inference_time:.6f} seconds/image ({average_inference_time * 1000:.3f} ms/image)")

---
## STEP 7. Report Classification Metrics

Calculate Accuracy, Precision, Recall, and F1-score on the test set predictions `y_pred`.

In [ ]:
# STEP 7. Report the classification metrics
# YOUR CODE HERE
# accuracy = ...
# precision = ...
# recall = ...
# f1 = ...

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

---
## STEP 8. Plot Confusion Matrix

Display the confusion matrix for test predictions.

In [ ]:
# STEP 8. Plot the confusion matrix
# YOUR CODE HERE
# matrix_values = ...
# confusion_matrix_display = ...
# confusion_matrix_display.plot(...)


---
## STEP 9. Show Classified Images

Display top 5 correctly classified images and top 5 incorrectly classified images using `show_images`.

In [ ]:
# STEP 9. Show 5 correctly/incorrectly classified images
# YOUR CODE HERE
# show_images(ground_truth=y_test, predictions=y_pred, img_test=img_test, correct=True, num_images=5)
# show_images(ground_truth=y_test, predictions=y_pred, img_test=img_test, correct=False, num_images=5)

---
## Reflection Questions

1. **Q1:** Why do we need a separate **Validation Set** to find $k$, instead of picking $k$ based on performance on the **Test Set**?

2. **Q2:** Why are 3D color histograms invariant to rotation and scale, but sensitive to lighting variations?

3. **Q3:** Why is KNN inference speed slow on large image datasets?